# 03 — 3-Line API, Differentiable Drift/PSF Recovery & Active Acquisition

This tutorial showcases the high-level **X-FUSE (`v1.0.0+`)** capabilities designed for real experimental workflows and live synchrotron/electron-microscopy beamtime:
1. **3-Line `XFuse.fit_predict` Interface**: Automatic normalization, spatial padding/cropping, multi-scale ViT feature extraction (`layers="multiscale"`), and frozen TTA feature caching.
2. **Differentiable Sub-Pixel Drift & Anisotropic PSF Learning**: Jointly compensate sample stage/beam drift (`align_guide="translation"`) and per-channel interaction-volume widths (`learn_psf=True`).
3. **Active Sparse-Slice & 2D ROI Steering**: Automatically schedule the next optimal $z$-slices (`suggest_next_slices`) or 2D re-scan sub-windows (`suggest_2d_rois`).
4. **Standardized Scientific I/O**: Export and reload `UncertaintyReport` archives in `.ome.tif` and `.h5` formats.

In [ ]:
from pathlib import Path
import tempfile

import numpy as np
from scipy.ndimage import shift as nd_shift
import torch

from x_fuse import (
    VolumetricFuser,
    XFuse,
    compute_segmentation_metrics,
    generate_correlative_phantom_2d,
    generate_correlative_phantom_3d,
    load_uncertainty_report,
    make_experimental_holdout,
    save_uncertainty_report,
    suggest_2d_rois,
    suggest_next_slices,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BACKBONE = "dinov2_vits14_reg" if DEVICE == "cuda" else "timm:vit_tiny_patch16_224"
PRETRAINED = DEVICE == "cuda"
print(f"Device={DEVICE}, Backbone={BACKBONE}, Pretrained={PRETRAINED}")

## 1. Sub-Pixel Drift & Learnable PSF Recovery via 3-Line `XFuse.fit_predict`
We simulate $+3.0$ HR pixels of horizontal specimen drift between the structural scan and a $4\times$ coarse 2-channel chemical map, then compare unaligned vs. drift-compensated X-FUSE.

In [ ]:
phantom = generate_correlative_phantom_2d(
    size=96, downsample_factor=4, phase_b_contrast=0.35, seed=7
)
gt_channels = np.stack(
    [(phantom.gt_labels == 1).astype(np.float32), (phantom.gt_labels == 2).astype(np.float32)],
    axis=0,
)
drifted_gt = np.stack([nd_shift(ch, (0.0, 3.0), order=1, mode="nearest") for ch in gt_channels])
_, drifted_guide, _ = make_experimental_holdout(
    phantom.hr_image, drifted_gt, factor=4, psf_sigma=2.0
)

model_aligned = XFuse(
    backbone=BACKBONE,
    layers="multiscale",
    align_guide="translation",
    fusion_method="spatial_mlp",
    loss_fn="forward_psf_bce",
    learn_psf=True,
    stride=8,
    pretrained=PRETRAINED,
    device=DEVICE,
)
report_2d = model_aligned.fit_predict(phantom.hr_image, drifted_guide, epochs=25)
metrics = compute_segmentation_metrics(report_2d.hard_mask[0].cpu().numpy(), phantom.gt_labels)
print("Aligned mIoU:", round(metrics["iou"], 4))
print("Registration & PSF diagnostics:", report_2d.diagnostics)

## 2. Targeted 2D ROI Re-Acquisition & OME-TIFF / HDF5 Export
Using `suggest_2d_rois`, we locate non-overlapping sub-windows with highest combined uncertainty + physical residual, and export the full multi-channel report.

In [ ]:
rois = suggest_2d_rois(report_2d, roi_size=(24, 24), k=2)
for idx, roi in enumerate(rois, 1):
    print(f"ROI #{idx}: y=[{roi.y0}:{roi.y1}], x=[{roi.x0}:{roi.x1}], score={roi.score}")

with tempfile.TemporaryDirectory() as tmpdir:
    ome_path = save_uncertainty_report(
        report_2d,
        Path(tmpdir) / "sample_fused.ome.tif",
        channel_names=["matrix", "phase_A", "phase_B"],
        pixel_size_um=0.025,
    )
    reloaded = load_uncertainty_report(ome_path)
    print("Exported OME-TIFF stack shape:", reloaded["stack"].shape)

## 3. Closed-Loop Active Sparse-Slice 3D Tomography
Starting from just 2 boundary chemical slices in a 3D structural tomogram, `suggest_next_slices` ranks the most informative interior $z$-slices to scan next.

In [ ]:
phantom_3d = generate_correlative_phantom_3d(shape=(20, 48, 48), downsample_factor=4, seed=11)
vol_fuser = VolumetricFuser(
    XFuse(backbone=BACKBONE, stride=8, pretrained=PRETRAINED, device=DEVICE)
)
initial_slices = {2: phantom_3d.coarse_guide_3d[:, 0], 17: phantom_3d.coarse_guide_3d[:, 4]}
vol_fuser.fit_sparse_slices(phantom_3d.hr_volume, sparse_guides=initial_slices, epochs=10)
vol_report = vol_fuser.predict_volume(phantom_3d.hr_volume, mode="triplane")

rec = suggest_next_slices(vol_report, acquired_indices=list(initial_slices), k=3, min_spacing=3)
print("Recommended next z-slices:", rec.recommended_indices)
print("Expected uncertainty gain:", rec.expected_uncertainty_gain)